# 🎭 MuseTalk — Coach IA Sèche 10 Semaines
**Génère des vidéos lip-sync avec un avatar IA**

⚠️ **IMPORTANT** : Va dans `Exécution > Modifier le type d'exécution > GPU T4` avant de lancer !

In [ ]:
#@title 1️⃣ Installation (3-5 min, une seule fois)
!git clone https://github.com/TMElyralab/MuseTalk.git
%cd MuseTalk
!pip install -r requirements.txt -q
!pip install edge-tts -q

# Télécharger les modèles
!python -c "
from huggingface_hub import snapshot_download
snapshot_download(repo_id='TMElyralab/MuseTalk', local_dir='./models/musetalk')
"
print('✅ Installation terminée !')

In [ ]:
#@title 2️⃣ Upload ton image de coach
from google.colab import files
import shutil

print('📸 Upload ton image de coach (coach-avatar.png) :')
uploaded = files.upload()
for fn in uploaded:
    shutil.move(fn, f'./data/images/{fn}')
    IMAGE_PATH = f'./data/images/{fn}'
    print(f'✅ Image sauvegardée : {IMAGE_PATH}')

In [ ]:
#@title 3️⃣ Générer les audios des 10 scripts viraux
import edge_tts
import asyncio
import os

os.makedirs('./data/audio', exist_ok=True)

SCRIPTS = [
    ("01-google-symptomes", "J'ai googlé mes symptômes. Fatigué après chaque repas. Ventre qui gonfle. Envie de sucre à 16h. Google m'a dit prédiabète. Mon médecin aussi. Mais au lieu de paniquer, j'ai juste changé ce que je mange. Pas un régime. Un vrai plan adapté à mon corps. 10 semaines plus tard. Plus de coups de barre. 8 kilos en moins. Et mon médecin qui comprend pas. Commente GLYCÉMIE si tu veux savoir ce que j'ai changé."),
    ("02-medecin-choque", "Mon médecin m'a dit de prendre des médicaments. Pour mon cholestérol. À 42 ans. J'ai dit non. Donnez-moi 10 semaines. J'ai pas fait de sport. J'ai pas compté les calories. J'ai juste mangé les bons aliments. Au bon moment. 10 semaines plus tard, mon bilan sanguin était parfait. Mon médecin m'a demandé ce que j'avais fait. Je lui ai montré mon assiette. Commente ton âge, je te dis par quoi commencer."),
    ("03-arrete-petit-dej", "J'ai arrêté le petit-déjeuner classique. Pain, confiture, jus d'orange. Mon nutritionniste m'a expliqué pourquoi j'avais faim à 10h. Pic de glycémie. Puis crash. Puis fringale. Depuis 10 semaines je mange autrement le matin. Plus jamais faim avant midi. J'ai perdu 6 kilos sans effort. Commente MATIN si tu veux savoir ce que je mange maintenant."),
    ("04-balance-mentait", "Ma balance me mentait. 85 kilos depuis 3 ans. Je mangeais moins. Je bougeais plus. Rien ne changeait. Puis j'ai compris. C'était pas les calories. C'était l'inflammation. En 10 semaines j'ai changé ce que je mange. Pas combien. Moins 9 kilos. Et pour la première fois, c'est resté. Commente ton poids, je t'explique pourquoi ça bloque."),
    ("05-ventre-gonfle", "Ventre gonflé après chaque repas. Ballonnements. Fatigue. Je pensais que c'était normal. Mon médecin aussi. Puis j'ai découvert que 3 aliments que je mangeais tous les jours causaient tout ça. En 10 semaines j'ai tout changé. Ventre plat. Énergie stable. Digestion parfaite. Commente VENTRE si tu veux la liste des 3 aliments."),
    ("06-triglycerides", "Triglycérides trop hauts. Mon médecin voulait me mettre sous statines. J'ai demandé 3 mois. Il m'a donné 2. En 8 semaines mes triglycérides ont baissé de 40 pourcent. Sans médicaments. Juste en changeant mon alimentation. Mon médecin n'en revenait pas. Commente BILAN si tu veux mon plan."),
    ("07-regime-yoyo", "J'ai fait tous les régimes. Weight Watchers. Keto. Jeûne intermittent. Je perdais. Je reprenais. À chaque fois plus. Puis j'ai compris que le problème c'était pas ma volonté. C'était mon métabolisme. En 10 semaines j'ai réparé ça. Moins 11 kilos. Et ça fait 6 mois que c'est stable. Commente YOYO si t'en as marre de reprendre."),
    ("08-glycemie-cachee", "Ta glycémie te tue en silence. Tu manges bien. Tu fais du sport. Mais t'es fatigué. Tu stockes du gras. Tu comprends pas. J'étais pareil. Puis j'ai mesuré ma glycémie après les repas. Le choc. Des pics à 180. Avec des aliments soi-disant sains. 10 semaines de changements. Glycémie stable. 7 kilos en moins. Commente GLYCÉMIE pour comprendre."),
    ("09-tour-de-taille", "Mon tour de taille disait plus que ma balance. 102 centimètres. Zone rouge. Risque cardio. Risque diabète. En 10 semaines je suis passé à 89. Pas avec du cardio. Pas avec un régime. Juste en mangeant les bons aliments aux bons moments. Commente ta taille en centimètres, je te dis où t'en es."),
    ("10-50-ans", "Après 50 ans perdre du poids c'est pas pareil. Le métabolisme ralentit. Les hormones changent. Les régimes classiques marchent plus. J'ai trouvé une méthode adaptée à mon âge. En 10 semaines j'ai perdu 8 kilos. Sans me priver. Sans forcer. En mangeant mieux pas moins. Commente ton âge je t'envoie le plan adapté.")
]

async def generate_all():
    for name, text in SCRIPTS:
        path = f'./data/audio/{name}.mp3'
        c = edge_tts.Communicate(text, 'fr-FR-HenriNeural')
        await c.save(path)
        print(f'✅ {name}.mp3')

await generate_all()
print(f'\n🎤 {len(SCRIPTS)} audios générés !')

In [ ]:
#@title 4️⃣ Générer les 10 vidéos lip-sync 🎬
import subprocess
import glob

os.makedirs('./results', exist_ok=True)

audio_files = sorted(glob.glob('./data/audio/*.mp3'))
print(f'🎬 Génération de {len(audio_files)} vidéos...\n')

for i, audio in enumerate(audio_files, 1):
    name = os.path.splitext(os.path.basename(audio))[0]
    output = f'./results/{name}.mp4'
    print(f'[{i}/{len(audio_files)}] {name}...')
    
    result = subprocess.run([
        'python', '-m', 'scripts.inference',
        '--driven_audio', audio,
        '--source_image', IMAGE_PATH,
        '--result_dir', './results',
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f'  ✅ OK')
    else:
        print(f'  ❌ Erreur: {result.stderr[-200:]}')

print(f'\n🎉 Terminé ! Vidéos dans ./results/')

In [ ]:
#@title 5️⃣ Télécharger toutes les vidéos 📥
import glob
from google.colab import files

# Zip tout
!cd results && zip -r /content/tiktok_videos.zip *.mp4

files.download('/content/tiktok_videos.zip')
print('📥 Téléchargement lancé !')